## LexisNexis 1B - Fraudulent Insurance Claim Detection

### Baseline Models (Logistic Regression)

#### Mukundan Thanigaivelan

### Setup

In [ ]:
# Data Wrangling
import pandas as pd
import numpy as np
import re

# Data visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Machine learning
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, average_precision_score, f1_score

### Data Inspection and Wrangling

In [ ]:
# Load in data
df = pd.read_csv('fraud_oracle.csv')
df.shape

(15420, 33)

In [ ]:
df.head()

,Month,WeekOfMonth,DayOfWeek,Make,AccidentArea,DayOfWeekClaimed,MonthClaimed,WeekOfMonthClaimed,Sex,MaritalStatus,...,AgeOfVehicle,AgeOfPolicyHolder,PoliceReportFiled,WitnessPresent,AgentType,NumberOfSuppliments,AddressChange_Claim,NumberOfCars,Year,BasePolicy
0,Dec,5,Wednesday,Honda,Urban,Tuesday,Jan,1,Female,Single,...,3 years,26 to 30,No,No,External,none,1 year,3 to 4,1994,Liability
1,Jan,3,Wednesday,Honda,Urban,Monday,Jan,4,Male,Single,...,6 years,31 to 35,Yes,No,External,none,no change,1 vehicle,1994,Collision
2,Oct,5,Friday,Honda,Urban,Thursday,Nov,2,Male,Married,...,7 years,41 to 50,No,No,External,none,no change,1 vehicle,1994,Collision
3,Jun,2,Saturday,Toyota,Rural,Friday,Jul,1,Male,Married,...,more than 7,51 to 65,Yes,No,External,more than 5,no change,1 vehicle,1994,Liability
4,Jan,5,Monday,Honda,Urban,Tuesday,Feb,2,Female,Single,...,5 years,31 to 35,No,No,External,none,no change,1 vehicle,1994,Collision


In [ ]:
# Columns
df.columns

Index(['Month', 'WeekOfMonth', 'DayOfWeek', 'Make', 'AccidentArea',
       'DayOfWeekClaimed', 'MonthClaimed', 'WeekOfMonthClaimed', 'Sex',
       'MaritalStatus', 'Age', 'Fault', 'PolicyType', 'VehicleCategory',
       'VehiclePrice', 'FraudFound_P', 'PolicyNumber', 'RepNumber',
       'Deductible', 'DriverRating', 'Days_Policy_Accident',
       'Days_Policy_Claim', 'PastNumberOfClaims', 'AgeOfVehicle',
       'AgeOfPolicyHolder', 'PoliceReportFiled', 'WitnessPresent', 'AgentType',
       'NumberOfSuppliments', 'AddressChange_Claim', 'NumberOfCars', 'Year',
       'BasePolicy'],
      dtype='object')

In [ ]:
# Number of null values
df.isna().sum().sum()

np.int64(0)

In [ ]:
# Inspect data types
df.dtypes.sort_values()

,0
PolicyNumber,int64
RepNumber,int64
Year,int64
FraudFound_P,int64
Age,int64
Deductible,int64
WeekOfMonthClaimed,int64
DriverRating,int64
WeekOfMonth,int64
PoliceReportFiled,object


In [ ]:
# Drop columns with identifiers
df = df.drop(columns = ['PolicyNumber', 'RepNumber'])

### Split data for training

In [ ]:
# Separate features and target
X = df.drop(columns = ['FraudFound_P'])
y = df['FraudFound_P']

In [ ]:
# Split before fitting any learned preprocessing
X_train, X_test, y_train, y_test = train_test_split(
  X, y,
  test_size = 0.20,
  random_state = 42,
  stratify = y
)

### Dummy Classifier

In [ ]:
dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_train, y_train)

dummy_pred = dummy.predict(X_test)
dummy_prob = dummy.predict_proba(X_test)[:, 1]

print("Dummy accuracy:", accuracy_score(y_test, dummy_pred))
print("Dummy precision:", precision_score(y_test, dummy_pred, zero_division=0))
print("Dummy recall:", recall_score(y_test, dummy_pred, zero_division=0))
print("Dummy F1:", f1_score(y_test, dummy_pred, zero_division=0))
print("Dummy PR-AUC:", average_precision_score(y_test, dummy_prob))

Dummy accuracy: 0.9400129701686122
Dummy precision: 0.0
Dummy recall: 0.0
Dummy F1: 0.0
Dummy PR-AUC: 0.05998702983138781


### Logistic Regression Model \#1 (Baseline)

- Treating all columns with text as categories, even ones with numerical ranges (good for a first baseline)

In [ ]:
# Identify feature types
numeric_cols = X_train.select_dtypes(
  include=["number"]
).columns.tolist()

categorical_cols = X_train.select_dtypes(
  include=["object", "category", "bool", "string"]
).columns.tolist()

In [ ]:
# Numeric columns: scale
numeric_pipeline = Pipeline([
  ("scaler", StandardScaler())
])

# Categorical columns: one-hot encode
categorical_pipeline = Pipeline([
  ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

In [ ]:
# Apply the appropriate transformations
preprocessor = ColumnTransformer([
  ("numeric", numeric_pipeline, numeric_cols),
  ("categorical", categorical_pipeline, categorical_cols)
])

In [ ]:
# Combine preprocessing and logistic regression
model = Pipeline([
  ("preprocessor", preprocessor),
  ("logistic", LogisticRegression(
    max_iter=3000,
    class_weight="balanced",
    random_state=42
  ))
])

In [ ]:
# Train model and predict
model.fit(X_train, y_train)

pred = model.predict(X_test)
prob = model.predict_proba(X_test)[:, 1]

In [ ]:
# Evaluate
print(f"Accuracy:  {accuracy_score(y_test, pred):.4f}")
print(f"Precision: {precision_score(y_test, pred, zero_division=0):.4f}")
print(f"Recall:    {recall_score(y_test, pred, zero_division=0):.4f}")
print(f"F1-score:  {f1_score(y_test, pred, zero_division=0):.4f}")
print(f"PR-AUC:    {average_precision_score(y_test, prob):.4f}")

Accuracy:  0.6420
Precision: 0.1315
Recall:    0.8865
F1-score:  0.2291
PR-AUC:    0.1564


### Convert text-based columns with numerical information to numbers

In [ ]:
cols_to_inspect = [
  "AgeOfPolicyHolder",
  "AgeOfVehicle",
  "Days_Policy_Accident",
  "Days_Policy_Claim",
  "PastNumberOfClaims",
  "NumberOfSuppliments",
  "NumberOfCars",
  "AddressChange_Claim",
  "VehiclePrice",
]

for col in cols_to_inspect:
  print(f'{df[col].value_counts(dropna=False).sort_index()}\n')

AgeOfPolicyHolder
16 to 17     320
18 to 20      15
21 to 25     108
26 to 30     613
31 to 35    5593
36 to 40    4043
41 to 50    2828
51 to 65    1392
over 65      508
Name: count, dtype: int64

AgeOfVehicle
2 years          73
3 years         152
4 years         229
5 years        1357
6 years        3448
7 years        5807
more than 7    3981
new             373
Name: count, dtype: int64

Days_Policy_Accident
1 to 7             14
15 to 30           49
8 to 15            55
more than 30    15247
none               55
Name: count, dtype: int64

Days_Policy_Claim
15 to 30           56
8 to 15            21
more than 30    15342
none                1
Name: count, dtype: int64

PastNumberOfClaims
1              3573
2 to 4         5485
more than 4    2010
none           4352
Name: count, dtype: int64

NumberOfSuppliments
1 to 2         2489
3 to 5         2017
more than 5    3867
none           7047
Name: count, dtype: int64

NumberOfCars
1 vehicle      14316
2 vehicles       709
3 t

In [ ]:
"""
Cases: (AgeOfPolicyHolder, Days_Policy_Accident, PastNumberOfClaims, Days_Policy_Claim, NumberOfSuppliments, VehiclePrice)
<num> to <num>
more than <num>
less than <num>
none
over <num>
<num>

Cases: (AgeOfVehicle, NumberOfCars, AddressChange_Claim)
new
more than <num>
<num> years
<num> to <num>
<num> vehicle
<num> vehicles
<num> year
<num> to <num> years
no change
under 6 months
"""

'\nCases: (AgeOfPolicyHolder, Days_Policy_Accident, PastNumberOfClaims, Days_Policy_Claim, NumberOfSuppliments, VehiclePrice)\n<num> to <num>\nmore than <num>\nless than <num>\nnone\nover <num>\n<num>\n\nCases: (AgeOfVehicle, NumberOfCars, AddressChange_Claim)\nnew\nmore than <num>\n<num> years\n<num> to <num>\n<num> vehicle\n<num> vehicles\n<num> year\n<num> to <num> years\nno change\nunder 6 months\n'

In [ ]:
def parse_numeric_category(value, column):
  if pd.isna(value):
    return np.nan, "missing"

  s = str(value).strip().lower()

  # Special categories
  if s == "none":
    return 0.0, "none"

  if s == "new":
    return 0.0, "new"

  if s == "no change":
    return np.nan, "no_change"

  if s == "under 6 months":
    return 0.25, "under_6_months"

  # Ranges: "26 to 30", "3 to 4 vehicles", etc.
  found_match = re.fullmatch(
    r"(\d+(?:\.\d+)?)\s*to\s*(\d+(?:\.\d+)?)(?:\s+\w+)?",
    s
  )
  if found_match:
    low, high = map(float, found_match.groups())
    return (low + high) / 2, "range"

  # Open-ended values
  found_match = re.fullmatch(
    r"(more than|less than|over)\s+(\d+(?:\.\d+)?)(?:\s+\w+)?",
    s
  )
  if found_match:
    direction, number = found_match.groups()
    return float(number), direction.replace(" ", "_")

  # Exact values with optional units
  found_match = re.fullmatch(
    r"(\d+(?:\.\d+)?)(?:\s+\w+)?",
    s
  )
  if found_match:
    return float(found_match.group(1)), "exact"

  raise ValueError(
    f"Unrecognized value {value!r} in column {column!r}"
  )

In [ ]:
data = df.copy()

for col in cols_to_inspect:
  parsed = data[col].apply(
      lambda x: parse_numeric_category(x, col)
  )

  data[f"{col}_numeric"] = parsed.apply(lambda x: x[0])
  data[f"{col}_type"] = parsed.apply(lambda x: x[1])

  data.drop(columns=[col], inplace=True)

In [ ]:
data.T.head(15)

,0,1,2,3,4,5,6,7,8,9,...,15410,15411,15412,15413,15414,15415,15416,15417,15418,15419
Month,Dec,Jan,Oct,Jun,Jan,Oct,Feb,Nov,Dec,Apr,...,Nov,Nov,Nov,Nov,Nov,Nov,Nov,Nov,Dec,Dec
WeekOfMonth,5,3,5,2,5,4,1,1,4,3,...,3,4,4,4,4,4,5,5,1,2
DayOfWeek,Wednesday,Wednesday,Friday,Saturday,Monday,Friday,Saturday,Friday,Saturday,Tuesday,...,Sunday,Tuesday,Tuesday,Thursday,Friday,Friday,Thursday,Thursday,Monday,Wednesday
Make,Honda,Honda,Honda,Toyota,Honda,Honda,Honda,Honda,Honda,Ford,...,Dodge,Honda,Pontiac,Mazda,Chevrolet,Toyota,Pontiac,Toyota,Toyota,Toyota
AccidentArea,Urban,Urban,Urban,Rural,Urban,Urban,Urban,Urban,Urban,Urban,...,Urban,Rural,Urban,Urban,Urban,Urban,Urban,Rural,Urban,Urban
DayOfWeekClaimed,Tuesday,Monday,Thursday,Friday,Tuesday,Wednesday,Monday,Tuesday,Wednesday,Wednesday,...,Tuesday,Wednesday,Wednesday,Monday,Monday,Tuesday,Friday,Friday,Thursday,Thursday
MonthClaimed,Jan,Jan,Nov,Jul,Feb,Nov,Feb,Mar,Dec,Apr,...,Nov,Nov,Nov,Nov,Nov,Nov,Dec,Dec,Dec,Dec
WeekOfMonthClaimed,1,4,2,1,2,1,3,4,5,3,...,4,5,4,4,4,5,1,1,2,3
Sex,Female,Male,Male,Male,Female,Male,Male,Male,Male,Male,...,Female,Male,Female,Male,Male,Male,Male,Male,Female,Male
MaritalStatus,Single,Single,Married,Married,Single,Single,Married,Single,Single,Married,...,Single,Married,Single,Married,Single,Married,Married,Single,Married,Single


In [ ]:
data.to_csv('cleaned_fraud_oracle.csv', index = False)

### Logistic Regression Model \#2 (Baseline)

- Now incorporating info from columns with numerical text as numbers

In [ ]:
# Separate features and target
X = data.drop(columns = ['FraudFound_P'])
y = data['FraudFound_P']

In [ ]:
# Split before fitting any learned preprocessing
X_train, X_test, y_train, y_test = train_test_split(
  X, y,
  test_size = 0.20,
  random_state = 42,
  stratify = y
)

In [ ]:
# Identify feature types
numeric_cols = X_train.select_dtypes(
  include=["number"]
).columns.tolist()

categorical_cols = X_train.select_dtypes(
  include=["object", "category", "bool", "string"]
).columns.tolist()

In [ ]:
# Numeric columns: scale
numeric_pipeline = Pipeline([
  ("imputer", SimpleImputer(strategy="median")),
  ("scaler", StandardScaler())
])

# Categorical columns: one-hot encode
categorical_pipeline = Pipeline([
  ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

In [ ]:
# Apply the appropriate transformations
preprocessor = ColumnTransformer([
  ("numeric", numeric_pipeline, numeric_cols),
  ("categorical", categorical_pipeline, categorical_cols)
])

In [ ]:
# Combine preprocessing and logistic regression
model2 = Pipeline([
  ("preprocessor", preprocessor),
  ("logistic", LogisticRegression(
    max_iter=3000,
    class_weight="balanced",
    random_state=42
  ))
])

In [ ]:
# Train model and predict
model2.fit(X_train, y_train)

pred = model2.predict(X_test)
prob = model2.predict_proba(X_test)[:, 1]

In [ ]:
# Evaluate
print(f"Accuracy:  {accuracy_score(y_test, pred):.4f}")
print(f"Precision: {precision_score(y_test, pred, zero_division=0):.4f}")
print(f"Recall:    {recall_score(y_test, pred, zero_division=0):.4f}")
print(f"F1-score:  {f1_score(y_test, pred, zero_division=0):.4f}")
print(f"PR-AUC:    {average_precision_score(y_test, prob):.4f}")

Accuracy:  0.6394
Precision: 0.1313
Recall:    0.8919
F1-score:  0.2288
PR-AUC:    0.1630
